# 과제 3: 내 사진으로 재활용품 탐지기 만들기

## 과제 안내

과제 2에서는 사진 한 장에 물건 하나를 찍고, 그 물건이 무엇인지 맞히는 분류기를 만들었습니다. 이번에는 한 장에 물건을 여러 개 찍고, 물건마다 박스를 **직접 그려** 레이블을 만든 뒤, 사진 속 물건마다 위치와 종류를 함께 찾는 탐지기를 만듭니다.

| 클래스 번호 | 클래스 | 레이블 이름 | 예 |
|:---:|---|---|---|
| 0 | 캔 | `can` | 음료 캔, 통조림 캔 |
| 1 | 종이팩 | `carton` | 우유팩, 두유·주스 종이팩 |
| 2 | 페트병 | `pet` | 생수병, 음료 페트병 |

채점은 조교가 따로 찍고 박스를 쳐 둔 **비공개 테스트 사진**으로 합니다. 테스트 사진에는 한 장에 물건이 두 개 이상 나오고, 세 클래스가 섞여 있으며, 여러 장소에서 찍었습니다. 점수는 이론 9장의 mAP50으로 매깁니다.

실습 9의 YOLO는 학습까지 라이브러리가 한 줄로 해 주므로, 이 과제에서는 과제 2처럼 사전학습 모델의 마지막 부분만 바꿔 PyTorch로 직접 학습시킬 수 있는 **Faster R-CNN**을 씁니다. Faster R-CNN은 이론 9장의 대표 모델 표에 나온 2단계 탐지 모델이고, COCO로 학습한 가중치가 공개되어 있습니다.

**대응 자료:** [이론 9장 객체탐지](https://ralbu85.github.io/lecture_deeplearning/chapters/ch09.html)(바운딩 박스, IoU, 손실, mAP), [실습 9](https://ralbu85.github.io/lecture_deeplearning/labs/lab09.html)(레이블 파일의 형태), [과제 2](https://ralbu85.github.io/lecture_deeplearning/assignments/hw02/hw02.html)(동결과 마지막 층 교체).

**Colab 설정:** 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**를 고른 뒤 시작합니다.

### 사진 찍기

1. 한 장에 물건이 **두 개 이상** 나오게 찍습니다. 세 클래스를 섞어 놓고, 물건끼리 조금 겹치거나 일부가 가려진 사진도 찍습니다. 장소, 거리, 각도, 물건의 종류를 바꿔 가며 찍을수록 테스트 사진을 잘 맞힙니다.
2. 장수 제한은 없습니다. 처음에는 30장 정도로 시작하고, 7절의 결과를 보고 늘립니다.
3. 사진은 **JPG**로 찍습니다. 아이폰은 **설정 → 카메라 → 포맷 → 높은 호환성**을 고릅니다. 아이폰 기본 형식(HEIC)은 Make Sense에서 열리지 않습니다.
4. 직접 찍은 사진만 씁니다. 인터넷에서 내려받은 사진이나 다른 학생의 사진은 쓰지 않습니다. 채점할 때 사진의 촬영 정보와 학생 사이의 같은 사진을 확인합니다.

### 박스 치기 — Make Sense

[Make Sense](https://www.makesense.ai)는 가입 없이 브라우저에서 박스를 치는 무료 도구입니다. 사진은 내 컴퓨터의 브라우저 안에서만 열리고 서버로 올라가지 않습니다.

1. **Get Started**를 누르고 찍은 사진을 모두 끌어다 놓은 뒤 **Object Detection**을 고릅니다.
2. 레이블 이름을 `can`, `carton`, `pet` **순서대로** 세 개 만들고 프로젝트를 시작합니다. 이 순서가 클래스 번호 0, 1, 2가 됩니다.
3. 사진마다 물건마다 박스를 하나씩 그리고 레이블을 고릅니다. 일부가 가려지거나 사진 밖으로 잘린 물건은 보이는 부분까지 박스를 그립니다. 세 클래스가 아닌 물건에는 박스를 치지 않습니다.
4. 다 쳤으면 **Actions → Export Annotations**에서 **YOLO 형식의 zip**을 내려받습니다. Make Sense는 작업을 저장하지 않으므로, 탭을 닫거나 새로 고치기 전에 내보냅니다.

나중에 사진을 더 찍었으면, 새 사진만 Make Sense에서 같은 순서의 레이블로 박스를 치고 새 zip을 내려받습니다. 노트북은 폴더에 있는 zip을 모두 읽습니다.

### 드라이브에 올리기

구글 드라이브의 `내 드라이브`에 `hw03` 폴더를 만들고, 그 안에 `images` 폴더를 만들어 사진을 넣습니다. Make Sense에서 내려받은 zip은 이름을 바꾸지 않고 `hw03` 폴더에 그대로 넣습니다.

```
내 드라이브/hw03/
  images/                          찍은 사진 (JPG)
  labels_my-project-name_....zip   Make Sense에서 내려받은 zip
```

### 제출 규칙

1. 아래 첫 셀의 `STUDENT_ID`와 `NAME`을 본인 것으로 바꿉니다.
2. `____`를 모두 채운 뒤 **런타임 다시 시작 후 전체 실행**으로 위에서부터 끝까지 오류 없이 실행합니다.
3. 변수 이름과 함수 이름은 바꾸지 않습니다.
4. 마지막 "제출 전 확인" 셀이 `통과`를 출력하는지 확인합니다.
5. 아래 파일 세 개를 제출합니다. 앞의 두 개는 8절의 셀이 만들어 내려받게 해 줍니다.

| 파일 | 내용 |
|---|---|
| `hw03_학번.pt` | 학습한 박스 예측부의 가중치 |
| `hw03_학번_photos.zip` | 학습에 쓴 사진(크기를 줄인 것)과 레이블 |
| `hw03_학번.ipynb` | 출력이 포함된 노트북 (파일 → 다운로드 → .ipynb) |

6. 개인 과제이므로 다른 학생의 가중치 파일이나 사진, 레이블을 받아서 내지 않습니다. 가중치는 내가 찍고 박스를 친 사진으로만 학습하고, 가중치 숫자를 직접 고치지 않습니다.

### 채점 (100점)

| 항목 | 배점 |
|---|---:|
| 비공개 테스트 사진의 mAP50 | 70 |
| 가중치 파일이 정해진 모델에 그대로 들어가는가 | 10 |
| 사진·레이블 zip에 세 클래스의 박스가 모두 있는가 | 10 |
| 노트북의 빈칸을 모두 채우고 끝까지 실행했는가 | 10 |

mAP50에 따른 점수표는 테스트 사진을 준비한 뒤 공지합니다.

In [ ]:
STUDENT_ID = "20240000"   # 본인 학번으로 바꿉니다
NAME = "홍길동"            # 본인 이름으로 바꿉니다

## 1. 준비 — 제공 코드

In [ ]:
# ▶ 그대로 실행하세요 — 준비
import torch
import random

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
random.seed(42)
print(device)

Colab에서 GPU 런타임이면 `cuda`가 나옵니다.

## 2. 사진과 레이블 가져오기 — 제공 코드

Make Sense가 내보낸 레이블은 실습 9의 coco8과 같은 형태입니다. 사진마다 이름이 같은 `.txt` 파일이 하나씩 있고, 한 줄이 박스 하나이며, 한 줄의 다섯 숫자는 클래스 번호, 박스 중심의 $x$, 중심의 $y$, 너비, 높이입니다. 좌표와 크기는 사진의 너비와 높이로 나눈 0~1 값입니다.

휴대폰 사진은 크기가 커서 그대로 학습하면 느립니다. 그래서 아래 셀은 사진을 긴 변 1024로 줄여 Colab 안으로 복사하고, 사진마다 레이블을 짝지은 뒤, 사진의 5분의 1을 검증용으로 따로 떼어 둡니다. 레이블의 좌표는 0~1 값이므로 사진을 줄여도 그대로 씁니다. 박스가 하나도 없는 사진은 학습에 쓸 수 없으므로 건너뜁니다.

In [ ]:
# ▶ 그대로 실행하세요 — 구글 드라이브 연결
from google.colab import drive

drive.mount('/content/drive')
PHOTO_DIR = '/content/drive/MyDrive/hw03'

In [ ]:
# ▶ 그대로 실행하세요 — 사진을 줄이고 레이블과 짝지어 훈련용과 검증용으로 나누기
import os, csv, glob, shutil, zipfile, unicodedata
from PIL import Image, ImageOps

CLASSES = ['can', 'carton', 'pet']
DATA = './hw03_data'
nfc = lambda s: unicodedata.normalize('NFC', s)   # 맥에서 만든 한글 파일 이름도 같은 글자로 맞춘다

# Make Sense에서 내보낸 zip을 오래된 것부터 모두 읽는다. 같은 사진의 레이블이 여러 번 있으면 나중 것을 쓴다.
labels = {}
zips = sorted(glob.glob(os.path.join(PHOTO_DIR, '*.zip')), key=os.path.getmtime)
for zp in zips:
    with zipfile.ZipFile(zp) as z:
        for n in z.namelist():
            base = os.path.basename(n)
            if base.endswith('.txt') and not base.startswith('.'):
                labels[nfc(base[:-4])] = z.read(n).decode('utf-8', 'ignore')
if not zips:   # zip을 풀어서 올렸으면 hw03 폴더 아래의 .txt 파일을 읽는다
    for p in glob.glob(os.path.join(PHOTO_DIR, '**', '*.txt'), recursive=True):
        labels[nfc(os.path.basename(p)[:-4])] = open(p, encoding='utf-8', errors='ignore').read()
print('레이블 zip:', [os.path.basename(z) for z in zips], '레이블 파일', len(labels), '개')
assert labels, 'hw03 폴더에 Make Sense에서 내보낸 zip 파일이 없습니다.'

# 사진마다 레이블 줄을 읽어 확인한다
src = os.path.join(PHOTO_DIR, 'images')
assert os.path.isdir(src), f'{src} 폴더가 없습니다. 드라이브의 폴더 이름을 확인하세요.'
items, skipped = [], []
for name in sorted(n for n in os.listdir(src) if not n.startswith('.')):
    if os.path.splitext(name)[1].lower() not in ('.jpg', '.jpeg', '.png'):
        print('JPG나 PNG 사진이 아니라서 건너뜁니다:', name)
        continue
    lines = []
    for line in labels.get(nfc(os.path.splitext(name)[0]), '').splitlines():
        v = line.split()
        if len(v) != 5:
            continue
        c = int(float(v[0]))
        cx, cy, w, h = map(float, v[1:])
        assert 0 <= c < len(CLASSES), f'{name}: 클래스 번호가 {c}입니다. Make Sense에서 레이블을 can, carton, pet 세 개만 순서대로 만들었는지 확인하세요.'
        if w > 0 and h > 0:
            lines.append(f'{c} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    if lines:
        items.append((name, lines))
    else:
        skipped.append(name)
if skipped:
    print('박스가 없어서 건너뜁니다:', skipped)

# 사진을 섞어 5분의 1을 검증용으로 두고, 긴 변 1024로 줄여 저장한다
random.shuffle(items)
n_val = max(2, len(items) // 5)
shutil.rmtree(DATA, ignore_errors=True)
for split in ['train', 'val']:
    os.makedirs(os.path.join(DATA, split, 'images'))
    os.makedirs(os.path.join(DATA, split, 'labels'))
rows = []
for k, (name, lines) in enumerate(items):
    try:
        img = Image.open(os.path.join(src, name))
    except Exception:
        print('사진이 아니라서 건너뜁니다:', name)
        continue
    exif = img.getexif()                                  # 촬영 정보 (271은 제조사, 272는 기종)
    img = ImageOps.exif_transpose(img).convert('RGB')     # Make Sense가 보여 준 방향대로 세우고 RGB 세 채널로 맞춘다
    img.thumbnail((1024, 1024))                           # 긴 변이 1024보다 크면 1024로 줄인다
    split = 'val' if k < n_val else 'train'
    file = f'img_{k:04d}'
    img.save(os.path.join(DATA, split, 'images', file + '.jpg'), quality=90)
    with open(os.path.join(DATA, split, 'labels', file + '.txt'), 'w') as f:
        f.write('\n'.join(lines) + '\n')
    rows.append([split, file, nfc(name), len(lines), str(exif.get(271, '')), str(exif.get(272, ''))])

# 어떤 원본 사진을 어디에 넣었는지 기록해 둔다 (사진 zip에 함께 들어간다)
with open(os.path.join(DATA, 'manifest.csv'), 'w', newline='') as f:
    csv.writer(f).writerows([['split', 'file', 'original', 'boxes', 'maker', 'model']] + rows)

for split in ['train', 'val']:
    count = [0] * len(CLASSES)
    for p in glob.glob(os.path.join(DATA, split, 'labels', '*.txt')):
        for line in open(p):
            count[int(line.split()[0])] += 1
    print(split, '사진', len(os.listdir(os.path.join(DATA, split, 'images'))), '장, 박스', dict(zip(CLASSES, count)))
assert len(rows) >= 10, '박스를 친 사진이 10장 이상 있어야 합니다.'

훈련용과 검증용의 사진 수와 클래스별 박스 수가 나옵니다. 박스가 적은 클래스가 있으면 그 클래스의 물건을 더 넣어 찍습니다.

## 3. 데이터셋 — 채워 넣기 ①

Faster R-CNN은 사진 한 장의 정답을 두 가지 텐서가 든 사전으로 받습니다.

| 키 | 모양 | 내용 |
|---|---|---|
| `boxes` | (박스 수, 4) | 박스마다 왼쪽 위 $(x_1, y_1)$와 오른쪽 아래 $(x_2, y_2)$의 픽셀 좌표 |
| `labels` | (박스 수,) | 박스마다 클래스 번호. 0번은 배경이므로 레이블의 0, 1, 2에 1을 더한 1, 2, 3 |

그러므로 데이터셋은 레이블 한 줄의 중심·너비·높이(0~1 값)를 픽셀 단위의 두 모서리 좌표로 바꿔야 합니다. 사진의 너비와 높이는 `W`, `H`입니다.

훈련 사진에는 증강으로 좌우 뒤집기를 넣습니다. 분류에서는 사진만 뒤집으면 되었지만, 탐지에서는 박스도 함께 뒤집어야 박스가 물건을 계속 감쌉니다. 너비가 `W`인 사진을 좌우로 뒤집으면 $x$ 좌표가 $x$였던 점은 $W - x$로 옮겨 가고, 박스의 왼쪽과 오른쪽이 서로 바뀝니다.

In [ ]:
# ✏️ 직접 채워 보세요
from torch.utils.data import Dataset
from torchvision.transforms.functional import to_tensor

class RecycleDataset(Dataset):
    def __init__(self, split, train=False):
        self.dir = os.path.join(DATA, split)
        self.names = sorted(os.listdir(os.path.join(self.dir, 'images')))
        self.train = train                       # True이면 좌우 뒤집기 증강을 넣는다

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        name = self.names[i]
        img = Image.open(os.path.join(self.dir, 'images', name))
        W, H = img.size
        boxes, labels = [], []
        for line in open(os.path.join(self.dir, 'labels', name[:-4] + '.txt')):
            c, cx, cy, w, h = map(float, line.split())
            x1 = ____
            y1 = ____
            x2 = ____
            y2 = ____
            boxes.append([x1, y1, x2, y2])
            labels.append(int(c) + 1)             # 0번은 배경이므로 클래스 번호에 1을 더한다
        boxes = torch.tensor(boxes, dtype=torch.float32)
        if self.train and random.random() < 0.5:  # 훈련 사진의 절반은 좌우로 뒤집는다
            img = ImageOps.mirror(img)
            boxes[:, 0], boxes[:, 2] = ____, ____  # 뒤집은 사진에서 박스의 새 x1, x2
        target = {'boxes': boxes, 'labels': torch.tensor(labels, dtype=torch.int64)}
        return to_tensor(img), target

train_ds = RecycleDataset('train', train=True)
val_ds = RecycleDataset('val')
print(len(train_ds), len(val_ds))

훈련·검증 사진 수가 나와야 합니다.

아래 셀은 훈련 사진 여섯 장에 데이터셋이 돌려준 박스를 그립니다. 박스가 물건을 감싸고 이름이 맞아야 합니다. 몇 번 다시 실행해서 뒤집힌 사진에서도 박스가 맞는지 봅니다. 박스가 어긋나면 ①의 좌표 계산을, 이름이 틀리면 Make Sense에서 레이블을 만든 순서를 확인합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 데이터셋의 박스가 물건에 맞는지 확인
import matplotlib.pyplot as plt

# 사진(텐서) 위에 박스와 클래스 이름을 그린다. scores가 있으면 신뢰도도 적는다.
def draw(ax, img, boxes, labels, color, scores=None):
    ax.imshow(img.permute(1, 2, 0))                    # (채널, 높이, 너비) → (높이, 너비, 채널)
    for k, (box, label) in enumerate(zip(boxes.tolist(), labels.tolist())):
        x1, y1, x2, y2 = box
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=2))
        text = CLASSES[label - 1] if scores is None else f'{CLASSES[label - 1]} {scores[k]:.2f}'
        ax.text(x1, y1, text, color='white', backgroundcolor=color, fontsize=8)
    ax.axis('off')

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, i in zip(axes.flat, range(len(train_ds))):
    img, target = train_ds[i]
    draw(ax, img, target['boxes'], target['labels'], 'lime')
plt.show()

사진마다 크기와 박스 수가 달라서 여러 장을 텐서 하나로 쌓을 수 없습니다. 그래서 아래 로더는 한 묶음을 사진 목록과 정답 목록으로 돌려줍니다.

In [ ]:
# ▶ 그대로 실행하세요 — 데이터 로더
from torch.utils.data import DataLoader

def collate(batch):
    return tuple(zip(*batch))     # [(사진, 정답), ...] → (사진 목록, 정답 목록)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate)
images, targets = next(iter(train_loader))
print(len(images), targets[0]['boxes'].shape, targets[0]['labels'])

사진 4장과 첫 사진의 박스 모양, 클래스 번호가 나옵니다.

## 4. 모델 — 채워 넣기 ②

Faster R-CNN은 사진을 ResNet-50으로 읽어 특성 맵을 만들고, 그 위에서 물체가 있을 만한 후보 박스를 고른 뒤, 후보 박스마다 1024개의 숫자로 된 특성을 뽑습니다. 맨 끝의 **박스 예측부**는 이 1024개의 숫자로 후보 박스마다 두 가지를 출력합니다.

| 출력 | 개수 | 9장의 개념 |
|---|---|---|
| 클래스 점수 | 배경과 세 클래스, 4개 | 클래스 확률 (분류) |
| 박스 좌표 보정 | 클래스마다 4개, 16개 | 바운딩 박스 (회귀) |

COCO로 학습한 박스 예측부는 COCO의 클래스에 맞춰져 있습니다. 그러므로 과제 2에서 ResNet-18의 `fc`를 출력 3개짜리로 바꾼 것처럼, 모든 파라미터를 동결한 뒤 박스 예측부를 우리 클래스에 맞는 새것으로 바꿉니다. `FastRCNNPredictor(입력 수, 클래스 수)`가 박스 예측부를 만들고, 클래스 수에는 배경도 셉니다.

In [ ]:
# ✏️ 직접 채워 보세요
from torchvision.models import detection
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

weights = detection.FasterRCNN_ResNet50_FPN_Weights.COCO_V1   # COCO로 학습한 가중치
model = detection.fasterrcnn_resnet50_fpn(weights=____)
for p in model.parameters():
    p.requires_grad = ____
model.roi_heads.box_predictor = FastRCNNPredictor(____, ____)
model = model.to(device)

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(n_train)

학습할 파라미터는 `20500`개여야 합니다.

## 5. 평가 함수

### 5.1 IoU — 채워 넣기 ③

예측 박스가 맞았는지는 9장과 같이 정답 박스와의 IoU가 0.5 이상이고 클래스도 같은지로 판정합니다. 박스 `a`, `b`는 `[x1, y1, x2, y2]`입니다. 9장의 "좌표로 계산하기"처럼 교집합의 너비와 높이를 구하고, 두 박스가 떨어져 있으면 0을 돌려줍니다.

In [ ]:
# ✏️ 직접 채워 보세요
def iou(a, b):
    w = ____          # 교집합의 너비
    h = ____          # 교집합의 높이
    if w <= 0 or h <= 0:
        return 0.0
    inter = w * h
    union = ____
    return inter / union

print(round(iou([2, 1, 6, 4], [4, 2, 8, 5]), 2), iou([0, 0, 1, 1], [2, 2, 3, 3]))

9장의 칸 세기 예시이므로 `0.2`와 `0.0`이 나와야 합니다.

### 5.2 AP와 mAP50 — 제공 코드

아래 함수들은 9장의 정의대로 클래스마다 AP를 구하고 평균해 mAP50을 계산합니다. 예측 박스를 신뢰도가 높은 순서로 하나씩 더하면서 맞은 박스가 나올 때마다 그때의 정밀도를 더하고, 그 합을 정답 박스 수로 나눕니다. 정답 박스 하나에는 예측 박스 하나만 맞음으로 셉니다.

In [ ]:
# ▶ 그대로 실행하세요 — AP와 mAP50
# 사진마다 모델의 예측과 정답을 모은다
def predict(model, ds):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for images, targets in DataLoader(ds, batch_size=4, collate_fn=collate):
            out = model([img.to(device) for img in images])
            preds += [{k: v.cpu() for k, v in o.items()} for o in out]
            gts += list(targets)
    return preds, gts

# 예측 박스와 가장 많이 겹치는 정답 박스가 IoU 0.5 이상이고, 아직 다른 예측이 맞히지 않았으면 맞음이다
def match(box, truth, used):
    ious = [iou(box, t) for t in truth]
    if ious and max(ious) >= 0.5:
        j = ious.index(max(ious))
        if not used[j]:
            used[j] = True
            return True
    return False

# 클래스 번호 c의 AP
def average_precision(preds, gts, c):
    dets = []                                          # (신뢰도, 사진 번호, 박스)
    for i, p in enumerate(preds):
        for box, label, score in zip(p['boxes'].tolist(), p['labels'].tolist(), p['scores'].tolist()):
            if label == c:
                dets.append((score, i, box))
    dets.sort(key=lambda d: -d[0])                     # 신뢰도가 높은 순서로
    truth = [[b for b, l in zip(g['boxes'].tolist(), g['labels'].tolist()) if l == c] for g in gts]
    used = [[False] * len(t) for t in truth]
    n_truth = sum(len(t) for t in truth)
    hit, total = 0, 0.0
    for k, (score, i, box) in enumerate(dets, start=1):
        if match(box, truth[i], used[i]):
            hit += 1
            total += hit / k                           # 맞은 박스가 나올 때의 정밀도
    return total / n_truth if n_truth else None        # 정답 박스가 없는 클래스는 None

# 클래스별 AP와 그 평균 mAP50
def evaluate_map(model, ds):
    preds, gts = predict(model, ds)
    aps = {name: average_precision(preds, gts, c + 1) for c, name in enumerate(CLASSES)}
    valid = [v for v in aps.values() if v is not None]
    return sum(valid) / len(valid), aps

박스 예측부를 새로 바꾸었으므로, 학습 전의 검증 mAP50은 0에 가깝습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 학습 전의 검증 mAP50
print(evaluate_map(model, val_ds))

## 6. 학습 — 채워 넣기 ④

학습 모드의 Faster R-CNN은 사진 목록과 정답 목록을 받아 예측 대신 손실 네 가지를 사전으로 돌려줍니다.

In [ ]:
# ▶ 그대로 실행하세요 — 학습 모드에서 모델이 돌려주는 손실
model.train()
losses = model([img.to(device) for img in images],
               [{k: v.to(device) for k, v in t.items()} for t in targets])
for k, v in losses.items():
    print(k, round(v.item(), 4))

| 이름 | 뜻 | 학습에 쓰는가 |
|---|---|---|
| `loss_classifier` | 박스 예측부의 클래스 손실 (교차 엔트로피) | 씀 |
| `loss_box_reg` | 박스 예측부의 박스 손실 | 씀 |
| `loss_objectness` | 후보 박스를 고르는 부분의 손실 | 동결했으므로 쓰지 않음 |
| `loss_rpn_box_reg` | 후보 박스를 고르는 부분의 박스 손실 | 동결했으므로 쓰지 않음 |

9장에서 탐지 모델의 손실은 클래스 손실과 박스 손실의 합이었습니다. 이 모델의 박스 손실은 $1 - \text{IoU}$ 대신 좌표의 차이로 계산하지만, 예측 박스가 정답 박스에서 어긋날수록 커진다는 점은 같습니다.

아래 셀에서 옵티마이저에는 학습할 박스 예측부의 파라미터만 넘기고, 손실은 학습에 쓰는 두 손실의 합으로 만들고, 실습 6의 학습 함수처럼 기울기를 계산해 가중치를 고칩니다. 에폭 수는 검증 mAP50을 보면서 직접 정합니다. 훈련 사진이 40장이면 4장씩 10묶음이라 한 에폭에 가중치를 10번 고칩니다. 20 에폭에서 시작해 검증 mAP50이 더 오르지 않을 때까지 늘립니다.

In [ ]:
# ✏️ 직접 채워 보세요
optimizer = torch.optim.Adam(____, lr=0.001)

EPOCHS = ____
history = []
for epoch in range(EPOCHS):
    model.train()
    total = 0.0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        losses = model(images, targets)
        loss = ____
        optimizer.zero_grad()
        ____
        ____
        total += loss.item()
    val_map, val_ap = evaluate_map(model, val_ds)
    history.append(val_map)
    print(epoch, round(total / len(train_loader), 4), round(val_map, 4))

In [ ]:
# ▶ 그대로 실행하세요 — 검증 mAP50 그래프
plt.plot(range(1, len(history) + 1), history, marker='o')
plt.xlabel('epoch')
plt.ylabel('val mAP50')
plt.ylim(0, 1.05)
plt.show()

## 7. 검증 사진 결과 보기 — 제공 코드

아래 셀은 클래스별 AP와 함께, 신뢰도 0.5 이상인 박스만 남겼을 때의 정밀도와 재현율을 계산합니다. 정밀도가 낮으면 물건이 없는 곳이나 다른 물건에 박스를 그린 것이고, 재현율이 낮으면 찾지 못한 물건이 많은 것입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 클래스별 AP, 정밀도, 재현율
preds, gts = predict(model, val_ds)
for c, name in enumerate(CLASSES):
    ap = average_precision(preds, gts, c + 1)
    print(name, 'AP', None if ap is None else round(ap, 3))

hit = n_pred = n_truth = 0
for p, g in zip(preds, gts):
    keep = p['scores'] >= 0.5
    for c in range(1, len(CLASSES) + 1):
        boxes = [b for b, l in zip(p['boxes'][keep].tolist(), p['labels'][keep].tolist()) if l == c]
        truth = [b for b, l in zip(g['boxes'].tolist(), g['labels'].tolist()) if l == c]
        used = [False] * len(truth)
        hit += sum(match(b, truth, used) for b in boxes)
        n_pred += len(boxes)
        n_truth += len(truth)
print('신뢰도 0.5 이상: 정밀도', round(hit / max(n_pred, 1), 3), '재현율', round(hit / max(n_truth, 1), 3))

아래 셀은 검증 사진에 정답 박스(초록)와 신뢰도 0.5 이상인 예측 박스(빨강)를 함께 그립니다. 초록 박스에만 빨간 박스가 없으면 찾지 못한 물건이고, 빨간 박스만 있으면 틀린 박스입니다. 어떤 조건의 사진에서 틀리는지 보고 그런 사진을 더 찍어 박스를 친 뒤, 새 zip을 드라이브에 넣고 2절부터 다시 실행합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 검증 사진의 정답과 예측
n = min(len(val_ds), 6)
fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(15, 4.5 * ((n + 2) // 3)), squeeze=False)
for ax, i in zip(axes.flat, range(n)):
    img, target = val_ds[i]
    keep = preds[i]['scores'] >= 0.5
    draw(ax, img, target['boxes'], target['labels'], 'green')
    draw(ax, img, preds[i]['boxes'][keep], preds[i]['labels'][keep], 'red', preds[i]['scores'][keep].tolist())
for ax in axes.flat[n:]:
    ax.axis('off')
plt.show()

## 8. 제출 파일 만들기 — 제공 코드

박스 예측부의 가중치와 사진·레이블을 파일로 만들어 내려받습니다. 노트북은 메뉴 **파일 → 다운로드 → .ipynb**로 따로 내려받습니다.

In [ ]:
# ▶ 그대로 실행하세요 — 제출 파일 만들기
from google.colab import files

PT_FILE = f'hw03_{STUDENT_ID}.pt'
torch.save({'student_id': STUDENT_ID,
            'classes': CLASSES,
            'box_predictor': {k: v.cpu() for k, v in model.roi_heads.box_predictor.state_dict().items()}}, PT_FILE)
ZIP_FILE = shutil.make_archive(f'hw03_{STUDENT_ID}_photos', 'zip', DATA)

files.download(PT_FILE)
files.download(ZIP_FILE)
print(PT_FILE, ZIP_FILE)

## 제출 전 확인 — 제공 코드

아래 셀이 `통과`를 출력해야 합니다. 오류가 나면 해당 항목을 다시 확인합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 제출 전 확인
assert STUDENT_ID != "20240000" and NAME != "홍길동", "학번과 이름을 바꾸세요"
img, target = RecycleDataset('val')[0]
assert target['boxes'].shape[1] == 4 and bool((target['boxes'][:, 2] > target['boxes'][:, 0]).all()), "3절 ①의 박스 좌표를 확인하세요"
assert abs(iou([2, 1, 6, 4], [4, 2, 8, 5]) - 0.2) < 1e-9 and iou([0, 0, 1, 1], [2, 2, 3, 3]) == 0, "5.1절 ③의 IoU를 확인하세요"
assert n_train == 20500 and model.roi_heads.box_predictor.cls_score.out_features == 4, "4절 ②의 동결과 박스 예측부 교체를 확인하세요"
assert len(history) == EPOCHS, "6절 학습을 끝까지 실행하세요"
assert os.path.exists(PT_FILE) and os.path.exists(ZIP_FILE), "8절의 제출 파일 만들기를 실행하세요"
saved = torch.load(PT_FILE, weights_only=True)   # 채점기와 같은 방법으로 가중치 파일을 다시 읽는다
assert saved['classes'] == CLASSES and tuple(saved['box_predictor']['cls_score.weight'].shape) == (4, 1024), "8절의 제출 파일 만들기를 다시 실행하세요"
print("통과")